# Outlier CA phylogeny analysis, scaled up, with a non-replicator control

Repeats the border-versus-middle analysis of [`2026-05-11-hstrat-paper-outlier-plots.ipynb`](2026-05-11-hstrat-paper-outlier-plots.ipynb) with three changes.

1. **Larger run.** The outlier CA is run for $\sqrt{20}\times$ as many steps on a grid with $\sqrt{20}\times$ the side length. The pattern grows at constant speed, so this gives about $20\times$ the extant population (and so $20\times$ the phylogeny tips) of the reference run.
2. **Non-replicator control.** The same pipeline is applied to a second expanding CA, 3-4 Life (B34/S34), which grows from a small seed as a dense chaotic blob rather than by budding off discrete self-copying structures. It is run until its extant population matches the scaled-up outlier run.
3. **Persistence filter in place of a time cutoff.** Rather than cutting trees at a fixed time, every live cell is also sampled at an earlier snapshot, and trees are built from the early-snapshot cells whose lineages persist to the sampled regions of the final snapshot (see "Parameters" and "Persistence filter").

Unlike the reference notebook, which downloads phylogenies generated on GPU by [`pylib/track_outlier.ipynb`](../pylib/track_outlier.ipynb), both CAs are run here, on CPU, using `pylib.track_ca_fast.FastCATracker`. That class fuses the CA update and hstrat marker inheritance into one compiled pass over live cells, with the same inheritance semantics as `pylib.track_ca.MultiTracker` driven with `dstream_S=64`, `dstream_bitwidth=1`, and `add_random_noise=True` (see its module docstring).


In [ ]:
%load_ext watermark


In [ ]:
import math
import os
import typing

from downstream import dstream
import matplotlib as mpl
import matplotlib.path
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from phyloframe import _auxlib as pfa
from phyloframe import legacy as pfl
from scipy import ndimage as scipy_ndimage
from scipy import stats as scipy_stats
import seaborn as sns
from teeplot import teeplot as tp
from tqdm import tqdm

from pylib.persistence_filter import mark_persistent_tips
from pylib.track_ca_fast import (
    FastCATracker,
    decode_rule_table,
    make_lifelike_rule_table,
    parse_rle,
)
from pylib.tree import draw_scatter_tree


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get(
    "NOTEBOOK_NAME", "2026-11-27-outlier-plots-scaled-control"
)
teeplot_subdir


In [ ]:
pfa.seed_random(1)


## Parameters

The reference run took 15,000 steps on a 2,300 x 2,300 grid.

All live cells are sampled twice: `SNAPSHOT_LOOKBACK` steps before the end of the run, and at the end.
Border and middle regions are laid out on the final snapshot, as before.
The tree analyzed for a region then has one tip for each early-snapshot lineage that left descendants in that region, so tips stand for replicator units that persisted rather than for individual cells.

Markers use hybrid steady/tilted retention with time dilation: a marker is deposited only every `TIME_DILATION` steps, while inheritance still happens every step.
Steady retention alone, as used for the reference run, spaces strata about `num_steps / 64` apart, which is too coarse to relate snapshots this close together at this run length.
The marker budget is unchanged, at 64 one-bit differentia per cell.

Checked against exact ancestry tracking at reference scale (labeling every cell at the early snapshot and following the labels forward), the number of tips kept is within two of the number of early units that truly hold an ancestor, for every look-back tried from 200 to 6,000 steps.
Which early unit represents a lineage is less reliable this far back: with a 2,000 step look-back about 60% of kept tips sat in a unit that truly holds an ancestor, with the rest in closely related units.


In [ ]:
REFERENCE_NUM_STEPS = 15_000
REFERENCE_GRID_SIZE = 2_300

SCALE = 20  # target increase in extant population size
LINEAR_SCALE = math.sqrt(SCALE)

DSTREAM_ALGO = dstream.hybrid_0_steady_1_tilted_2_algo
TIME_DILATION = 16  # steps per marker deposit

# early snapshot is taken this many steps before the end of the run
SNAPSHOT_LOOKBACK = 2_000
# only nodes at least this many steps before the end enter the branch length
# plot (as in the reference notebook, somewhat deeper than the tips)
DELTA_LOOKBACK = 3_000

N_NULL_SAMPLES = 1_000  # random trees per null model


## Run CAs


In [ ]:
def run_ca(
    rule: np.ndarray,
    seed_pattern: np.ndarray,
    *,
    grid_size: int,
    num_steps: int,
    desc: str,
    seed: int = 1,
) -> pd.DataFrame:
    # Grow `seed_pattern` from the grid center, and reconstruct the phylogeny
    # of the cells alive after `num_steps`, along with those alive
    # `SNAPSHOT_LOOKBACK` steps earlier.
    grid = np.zeros((grid_size, grid_size), dtype=np.uint8)
    h, w = seed_pattern.shape
    oy, ox = (grid_size - h) // 2, (grid_size - w) // 2
    grid[oy : oy + h, ox : ox + w] = seed_pattern

    tracker = FastCATracker(
        rule, grid, DSTREAM_ALGO, seed=seed, time_dilation=TIME_DILATION
    )
    snapshot_step = num_steps - SNAPSHOT_LOOKBACK
    assert (
        snapshot_step // TIME_DILATION >= tracker.S
    ), "run too short to take a snapshot that early"
    with tqdm(total=num_steps, desc=desc, mininterval=30) as progress:
        for stop in snapshot_step, num_steps:
            while tracker.t < stop:
                chunk = min(500, stop - tracker.t)
                tracker.step(chunk)
                progress.update(chunk)
                progress.set_postfix(alive=tracker.num_alive, refresh=False)
            if stop == snapshot_step:
                tracker.snapshot()

    df = tracker.reconstruct_phylogeny()
    # ranks count deposits; convert to steps
    df["origin_time"] = df["hstrat_rank"].astype(np.int64) * TIME_DILATION
    df.attrs["num_steps"] = num_steps
    df["is_early"] = df["snapshot_step"] == snapshot_step
    df.attrs["snapshot_origin_time"] = int(
        df.loc[df["is_early"], "origin_time"].max()
    )
    df.attrs["seed_center"] = (oy + (h - 1) / 2, ox + (w - 1) / 2)
    return df


### Outlier CA

Same rule table and seed as the reference run.


In [ ]:
OUTLIER_RULE = decode_rule_table(
    "ERETQB4eHWkQ7xD4eYZosBQZFixOBHmtFeehExrKVhURLRAqGxeIlSO1JYZP6DRi69rop7TQCkvWTIag7kAS8g",
)
OUTLIER_SEED = parse_rle("$bo$3o$2bo!")
OUTLIER_NUM_STEPS = round(REFERENCE_NUM_STEPS * LINEAR_SCALE)
OUTLIER_GRID_SIZE = 8 * math.ceil(REFERENCE_GRID_SIZE * LINEAR_SCALE / 8)
OUTLIER_NUM_STEPS, OUTLIER_GRID_SIZE


In [ ]:
phylos = {}
phylos["outlier"] = run_ca(
    OUTLIER_RULE,
    OUTLIER_SEED,
    grid_size=OUTLIER_GRID_SIZE,
    num_steps=OUTLIER_NUM_STEPS,
    desc="outlier",
)


### Non-replicator control CA

3-4 Life (B34/S34), seeded with a 24 x 24 random soup.
Its population grows quadratically, reaching the size of the reference outlier run after about 1,200 steps, so its run length is scaled from there.
The front advances faster than the outlier's, so the control reaches the same population in fewer steps.


In [ ]:
CONTROL_RULE = make_lifelike_rule_table(birth=[3, 4], survive=[3, 4])
CONTROL_SEED = (np.random.default_rng(1).random((24, 24)) < 0.5).astype(
    np.uint8,
)
CONTROL_NUM_STEPS = round(1_200 * LINEAR_SCALE)
CONTROL_GRID_SIZE = 8 * math.ceil(0.55 * CONTROL_NUM_STEPS / 8)
CONTROL_NUM_STEPS, CONTROL_GRID_SIZE


In [ ]:
phylos["control"] = run_ca(
    CONTROL_RULE,
    CONTROL_SEED,
    grid_size=CONTROL_GRID_SIZE,
    num_steps=CONTROL_NUM_STEPS,
    desc="control",
)


In [ ]:
pd.DataFrame(
    {
        name: {
            "num steps": df.attrs["num_steps"],
            "num extant cells": df["extant"].sum(),
            "num early snapshot cells": df["is_early"].sum(),
            "num phylogeny nodes": len(df),
        }
        for name, df in phylos.items()
    },
)


## Label sampled regions

As in the reference notebook, extant cells (i.e., those in the final snapshot) are sampled from a "border" region along the expansion front and from a "middle" region in the interior.


In [ ]:
cmap = plt.get_cmap("Pastel1")
cmap


In [ ]:
def add_position_label(
    df: pd.DataFrame, vertices: list[tuple[float, float]], label_as: str
) -> None:
    if "position" not in df.columns:
        df["position"] = np.select(
            [df["extant"].astype(bool), df["is_early"]],
            ["unsampled", "early snapshot"],
            default="inner node",
        )
    in_shape = (
        matplotlib.path.Path(vertices).contains_points(
            np.vstack([df["row"].to_numpy(), df["col"].to_numpy()]).T
        )
        & df["extant"]
    )
    df["position"] = np.select([in_shape], [label_as], default=df["position"])


### Outlier CA

The outlier pattern is a square, tilted about 17.6 degrees off the grid axes, with a detached band of replicators running along one side.

- **Border**: that band. Its depth stays close to 200 cells as the pattern grows, so rather than scaling up the reference polygon (which would take in part of the interior), the region is laid `OUTLIER_BORDER_DEPTH` cells deep behind the front.
- **Middle**: the reference polygon, scaled about the seed in proportion to run length.


In [ ]:
REFERENCE_MIDDLE_SAMPLE = [(1600, 525), (1900, 625), (1475, 1800), (1175, 1700)]
OUTLIER_TILT = np.arctan2(1650, 5200)  # angle of border band off row axis
OUTLIER_BORDER_DEPTH = 210  # cells


def make_outlier_regions(df: pd.DataFrame) -> dict:
    center = np.array(df.attrs["seed_center"])
    factor = df.attrs["num_steps"] / REFERENCE_NUM_STEPS

    # unit vectors, as (row, col), along and across the border band
    along = np.array([np.cos(OUTLIER_TILT), np.sin(OUTLIER_TILT)])
    across = np.array([-np.sin(OUTLIER_TILT), np.cos(OUTLIER_TILT)])
    extant = df.loc[df["extant"], ["row", "col"]].to_numpy() - center
    front = (extant @ across).min()
    half_length = 1.5 * np.abs(extant @ along).max()
    border = [
        center + u * along + v * across
        for u, v in (
            (-half_length, front - 1),
            (half_length, front - 1),
            (half_length, front + OUTLIER_BORDER_DEPTH),
            (-half_length, front + OUTLIER_BORDER_DEPTH),
        )
    ]

    reference_center = (REFERENCE_GRID_SIZE - 1) / 2
    middle = [
        center + (np.array(vertex) - reference_center) * factor
        for vertex in REFERENCE_MIDDLE_SAMPLE
    ]
    return {"sampled border": border, "sampled middle": middle}


### Control CA

The control pattern is a rounded blob without a distinct border structure, so regions are laid out to be comparable in geometry and tip count to the outlier's.

- **Border**: cells within 8% of the front radius of the front, over a 150 degree arc (the outlier border is about 7.5% of the front distance deep).
- **Middle**: a rectangular strip through the center, perpendicular to the middle of the border arc.


In [ ]:
CONTROL_BORDER_DEPTH_FRAC = 0.08
CONTROL_BORDER_HALF_ARC = np.deg2rad(75)
CONTROL_MIDDLE_HALF_EXTENT_FRAC = (0.2, 0.6)  # (rows, cols)


def make_control_regions(df: pd.DataFrame) -> dict:
    extant = df.loc[df["extant"], ["row", "col"]].to_numpy()
    center = extant.mean(axis=0)
    drow, dcol = (extant - center).T
    angle = np.arctan2(drow, -dcol)  # zero is toward low col
    radius = np.hypot(drow, dcol)

    # trace front over arc
    edges = np.linspace(-CONTROL_BORDER_HALF_ARC, CONTROL_BORDER_HALF_ARC, 121)
    bins = np.digitize(angle, edges) - 1
    in_arc = (bins >= 0) & (bins < len(edges) - 1)
    front = np.zeros(len(edges) - 1)
    np.maximum.at(front, bins[in_arc], radius[in_arc])
    front_radius = front.mean()
    depth = CONTROL_BORDER_DEPTH_FRAC * front_radius

    def trace(radii: np.ndarray) -> list:
        mids = (edges[:-1] + edges[1:]) / 2
        return [
            center + r * np.array([np.sin(a), -np.cos(a)])
            for a, r in zip(mids, radii)
        ]

    border = trace(front + 1) + trace(front - depth)[::-1]

    half_rows, half_cols = (
        np.array(CONTROL_MIDDLE_HALF_EXTENT_FRAC) * front_radius
    )
    middle = [
        center + np.array(offset)
        for offset in (
            (-half_rows, -half_cols),
            (-half_rows, half_cols),
            (half_rows, half_cols),
            (half_rows, -half_cols),
        )
    ]
    return {"sampled border": border, "sampled middle": middle}


In [ ]:
regions = {
    "outlier": make_outlier_regions(phylos["outlier"]),
    "control": make_control_regions(phylos["control"]),
}
for name, df in phylos.items():
    for label, vertices in regions[name].items():
        add_position_label(df, vertices, label)

pd.DataFrame(
    {name: df["position"].value_counts() for name, df in phylos.items()},
)


## Analysis

Ported from the reference notebook, with the phylogeny passed in as an argument and the tree trimming changed as described next.


### Persistence filter

For each region, the phylogeny is first pruned to the region's final-snapshot cells plus every early-snapshot cell.
`mark_persistent_tips` then picks one early-snapshot tip for each lineage that carries on to the region: walking rootward from a final-snapshot tip, the first clade that also holds an early-snapshot tip marks where that lineage was last seen at the early snapshot, and one of the early tips there is kept.
The tree analyzed is the phylogeny of the kept tips.


In [ ]:
def get_delta_cutoff_time(df: pd.DataFrame) -> int:
    return df.attrs["snapshot_origin_time"] - (
        DELTA_LOOKBACK - SNAPSHOT_LOOKBACK
    )


_trimmed_tree_cache = {}


def prep_trimmed_tree(input_df: pd.DataFrame, key: str) -> pd.DataFrame:
    # Tree of early-snapshot tips whose lineages persist to cells sampled
    # for `key`.
    cache_key = (id(input_df), key)
    if cache_key not in _trimmed_tree_cache:
        df = input_df.copy()
        df["is_late"] = df["position"] == key
        df["extant"] = df["is_late"] | df["is_early"]
        df = pfl.alifestd_prune_extinct_lineages_asexual(df)
        df = pfl.alifestd_collapse_unifurcations(df)
        df = mark_persistent_tips(df, early="is_early", late="is_late")
        df["extant"] = df["is_persistent"]
        df = pfl.alifestd_prune_extinct_lineages_asexual(df)
        df = pfl.alifestd_collapse_unifurcations(df)
        _trimmed_tree_cache[cache_key] = df
    return _trimmed_tree_cache[cache_key].copy()


In [ ]:
def plot_regions(df: pd.DataFrame, ax=None):
    ax = sns.scatterplot(
        data=df[df["extant"]],
        x="col",
        y="row",
        s=0.5 / SCALE,
        linewidth=0,
        palette={
            "sampled border": cmap.colors[0],
            "sampled middle": cmap.colors[1],
            "unsampled": cmap.colors[-3],
        },
        hue="position",
        hue_order=["sampled border", "sampled middle", "unsampled"],
        rasterized=True,
        ax=ax,
    )
    ax.set_aspect("equal")
    ax.legend(title="position", markerscale=5 * LINEAR_SCALE)
    return ax


In [ ]:
def plot(
    df: pd.DataFrame,
    color_by: typing.Optional[str] = None,
    scatter_kws: dict = {},
    tree_kws: dict = {},
    **kwargs,
):
    return draw_scatter_tree(
        df,
        hue=color_by,
        scatter_kws=dict(
            alpha=0.8,
            edgecolor="none",
            legend=True,
        )
        | scatter_kws,
        scatter_shuffle=1,
        tree_kws=dict(
            edge=dict(
                color="gainsboro",
                linewidth=0.5,
            ),
            margins=-0.05,
            layout_angular=True,
        )
        | tree_kws,
        **kwargs,
    )


In [ ]:
def plot_colless_index_trees(input_df: pd.DataFrame, key: str, input_ax=None):
    df = prep_trimmed_tree(input_df, key)
    df[key] = True
    color = cmap.colors[0] if key == "sampled border" else cmap.colors[1]
    ax = plot(
        df,
        color_by=key,
        tree_kws={"edge": {"color": [color]}},
        scatter_kws={"palette": [color], "s": 4},
        collapse_unifurcations=False,
        ax=input_ax,
    )
    ax.get_legend().remove()


In [ ]:
TREE_GENERATORS = (pfl.alifestd_make_edge_split, pfl.alifestd_make_leaf_split)


def sample_colless_index_data(input_df: pd.DataFrame, n_samples: int):
    data = {
        key: {"source": [], "colless_index": []}
        for key in ("sampled border", "sampled middle")
    }
    observed = {}
    leaf_node_counts = {}

    for key in "sampled border", "sampled middle":
        df = prep_trimmed_tree(input_df, key)
        df = pfl.alifestd_mark_colless_like_index_mdm_asexual(df)
        observed[key] = df["colless_like_index_mdm"].iloc[0]
        leaf_node_count = pfl.alifestd_count_leaf_nodes(df)
        leaf_node_counts[key] = leaf_node_count
        for _ in tqdm(
            range(n_samples),
            desc=f"sampling trees for {key=}",
            mininterval=30,
        ):
            for f in TREE_GENERATORS:
                tree = f(leaf_node_count)
                tree = pfl.alifestd_mark_colless_like_index_mdm_asexual(tree)
                data[key]["source"].append(
                    f.__name__.removeprefix("alifestd_make_")
                )
                data[key]["colless_index"].append(
                    tree["colless_like_index_mdm"].iloc[0]
                )

    return data, observed, leaf_node_counts


In [ ]:
def plot_colless_index_data(
    sampled: dict, observed: dict, key: str, input_ax=None
):
    COLOR = "#464c5e"
    OBSERVED_COL_NAME = "observed"

    df = pd.DataFrame(sampled[key])
    lower_bound = df["colless_index"].max()
    upper_bound = df["colless_index"].min()
    for f in TREE_GENERATORS:
        real_name = f.__name__.removeprefix("alifestd_make_")
        lower_bound = min(
            lower_bound,
            df[df["source"] == real_name]["colless_index"].quantile(0.025),
        )
        upper_bound = max(
            upper_bound,
            df[df["source"] == real_name]["colless_index"].quantile(0.975),
        )
    color = cmap.colors[0] if key == "sampled border" else cmap.colors[1]
    ax = sns.boxplot(
        data=df,
        x="source",
        y="colless_index",
        showfliers=False,
        width=0.2,
        order=("leaf_split", "edge_split", OBSERVED_COL_NAME),
        boxprops={"facecolor": COLOR, "edgecolor": COLOR},
        whiskerprops={"color": COLOR},
        medianprops={"color": COLOR},
        capprops={"linewidth": 0},
        ax=input_ax,
    )
    ax = sns.scatterplot(
        data=pd.DataFrame(
            {
                "colless_index": [observed[key]],
                "source": [OBSERVED_COL_NAME],
            }
        ),
        x="source",
        y="colless_index",
        color=color,
        ax=ax,
    )
    y_max = ax.get_ylim()[1]
    ax.set_ylim([0, y_max * 1.25])
    ax.axhspan(lower_bound, upper_bound, color=f"{COLOR}20")
    ax.yaxis.tick_right()
    ax.yaxis.set_label_position("right")
    return ax


In [ ]:
def prep_origin_time_delta(input_df: pd.DataFrame) -> pd.DataFrame:
    cutout_dfs = [
        prep_trimmed_tree(input_df, key).pipe(
            pfl.alifestd_mark_origin_time_delta_asexual
        )
        for key in ("sampled border", "sampled middle")
    ]
    return (
        pd.concat(cutout_dfs, keys=["border", "middle"])
        .reset_index(level=0)
        .rename(columns={"level_0": "source"})
    )


def plot_origin_time_delta(input_df: pd.DataFrame, input_ax=None):
    delta_cutoff_time = get_delta_cutoff_time(input_df)
    df = prep_origin_time_delta(input_df)
    original_df = df.copy()
    for src in ("border", "middle"):
        original_df[f"{src}_marker_type"] = "."
        original_df[f"{src}_hue"] = "#ffffff"
    markers = "osDv"

    for i, src in enumerate(("border", "middle")):
        for j, m in enumerate(markers):
            plot_df = df[
                (df["source"] == src) & (df["origin_time"] < delta_cutoff_time)
            ].copy()
            plot_df["marker_col"] = range(len(plot_df))
            plot_df.loc[
                plot_df["marker_col"] % len(markers) != j, "origin_time"
            ] = 1e10
            origin_times = sorted(plot_df["origin_time"].unique())
            colors = sns.dark_palette(
                cmap.colors[i], n_colors=len(origin_times)
            )[1:] + [(0, 0, 0, 0)]
            for id in plot_df[plot_df["marker_col"] % len(markers) == j]["id"]:
                is_id = (original_df["id"] == id) & (
                    original_df["source"] == src
                )
                original_df.loc[is_id, f"{src}_marker_type"] = m
                color_to_be_plotted = colors[
                    origin_times.index(
                        original_df.loc[is_id, "origin_time"].values[0]
                    )
                ]
                original_df.loc[is_id, f"{src}_hue"] = mpl.colors.rgb2hex(
                    color_to_be_plotted
                )
            ax = sns.swarmplot(
                data=plot_df,
                y="origin_time_delta",
                x="source",
                hue="origin_time",
                palette=colors,
                order=("border", "middle"),
                legend=False,
                marker=m,
                ax=input_ax,
            )

    ax.tick_params(
        axis="y",
        which="both",
        labelleft=False,
        left=False,
        labelright=True,
        right=True,
    )
    ax.yaxis.set_label_position("right")
    return original_df, ax


In [ ]:
def plot_origin_time_delta_trees(
    input_df: pd.DataFrame, marker_info_df: pd.DataFrame, key: str, input_ax=None
):
    df = prep_trimmed_tree(input_df, key)

    src = key[-6:]
    color = cmap.colors[0] if src == "border" else cmap.colors[1]
    is_src = marker_info_df["source"] == src
    markers = list(marker_info_df[is_src][f"{src}_marker_type"])
    palette = list(marker_info_df[is_src][f"{src}_hue"])

    return plot(
        df,
        "id",
        tree_kws={
            "edge": {
                "color": [color],
                "linewidth": 0.01,
            }
        },
        scatter_kws={"markers": markers, "palette": palette, "legend": False},
        collapse_unifurcations=False,
        style="id",
        ax=input_ax,
    )


In [ ]:
def plot_persistent_tips(df: pd.DataFrame, ax=None):
    # Early snapshot, with tips kept by the persistence filter highlighted.
    if ax is None:
        ax = plt.gca()
    early_df = df[df["is_early"]]
    ax.scatter(
        early_df["col"],
        early_df["row"],
        s=0.5 / SCALE,
        linewidth=0,
        color=cmap.colors[-3],
        rasterized=True,
    )
    for i, key in enumerate(("sampled border", "sampled middle")):
        tree_df = prep_trimmed_tree(df, key)
        tips_df = tree_df[tree_df["is_persistent"]]
        ax.scatter(
            tips_df["col"],
            tips_df["row"],
            s=6,
            linewidth=0,
            color=sns.dark_palette(cmap.colors[i], n_colors=4)[2],
            label=key.replace("sampled", "persists to"),
        )
    ax.set_aspect("equal")
    ax.set_xlabel("col")
    ax.set_ylabel("row")
    ax.legend(title="early snapshot tip", markerscale=2)
    return ax


### Figure assembly


In [ ]:
null_samples = {}


def show_regions(name: str) -> None:
    with tp.teed(
        plot_regions,
        phylos[name],
        teeplot_outattrs={"ca": name},
        teeplot_subdir=teeplot_subdir,
    ):
        ...


def show_persistent_tips(name: str) -> None:
    with tp.teed(
        plot_persistent_tips,
        phylos[name],
        teeplot_outattrs={"ca": name},
        teeplot_subdir=teeplot_subdir,
    ):
        ...


def show_colless(name: str) -> None:
    with tp.teed(
        plt.subplots,
        2,
        2,
        tight_layout=True,
        figsize=(5, 5),
        gridspec_kw={"width_ratios": [1, 1.3]},
        teeplot_outattrs={"ca": name, "viz": "plot_colless_data"},
        teeplot_subdir=teeplot_subdir,
    ) as (fig, axes):
        df = phylos[name]
        plot_colless_index_trees(df, "sampled border", axes[0][0])
        plot_colless_index_trees(df, "sampled middle", axes[1][0])
        null_samples[name] = sample_colless_index_data(df, N_NULL_SAMPLES)
        sampled, observed, __ = null_samples[name]
        plot_colless_index_data(sampled, observed, "sampled border", axes[0][1])
        axes[0][1].set_xlabel(None)
        axes[0][1].set_xticks([])
        plot_colless_index_data(sampled, observed, "sampled middle", axes[1][1])


def show_branch_lengths(name: str) -> None:
    with tp.teed(
        plt.subplot_mosaic,
        [["border", "wait_times"], ["middle", "wait_times"]],
        tight_layout=True,
        figsize=(5, 5),
        gridspec_kw={"width_ratios": [1, 1.5]},
        teeplot_outattrs={"ca": name, "viz": "plot_branch_length_data"},
        teeplot_subdir=teeplot_subdir,
    ) as (fig, axes):
        df = phylos[name]
        marker_info_df, __ = plot_origin_time_delta(df, axes["wait_times"])
        plot_origin_time_delta_trees(
            df, marker_info_df, "sampled border", axes["border"]
        )
        plot_origin_time_delta_trees(
            df, marker_info_df, "sampled middle", axes["middle"]
        )


## Outlier CA

### (a) Sampled regions

Extant cells, with those sampled for "border" (red) and "middle" (blue) highlighted.


In [ ]:
show_regions("outlier")


Early snapshot (2,000 steps before the end), with the tips kept by the persistence filter for each region highlighted.


In [ ]:
show_persistent_tips("outlier")


### (b) Tree imbalance

Trees of persistent early-snapshot tips for each region (left), and their Colless-like imbalance compared against random trees with the same number of leaves generated by edge splitting and leaf splitting (right).
Shaded band spans the 2.5th to 97.5th percentiles of the null models.


In [ ]:
show_colless("outlier")


### (c) Branch lengths

Inner branch lengths of the same trees (left), as a swarmplot with markers matched to tree nodes (right).


In [ ]:
show_branch_lengths("outlier")


## Non-replicator control CA

### (a) Sampled regions

Extant cells, with those sampled for "border" (red) and "middle" (blue) highlighted.


In [ ]:
show_regions("control")


Early snapshot (2,000 steps before the end), with the tips kept by the persistence filter for each region highlighted.


In [ ]:
show_persistent_tips("control")


### (b) Tree imbalance

Trees of persistent early-snapshot tips for each region (left), and their Colless-like imbalance compared against random trees with the same number of leaves generated by edge splitting and leaf splitting (right).
Shaded band spans the 2.5th to 97.5th percentiles of the null models.


In [ ]:
show_colless("control")


### (c) Branch lengths

Inner branch lengths of the same trees (left), as a swarmplot with markers matched to tree nodes (right).


In [ ]:
show_branch_lengths("control")


## Tips and replicator units

Replicator units in the outlier CA are counted as connected clusters of cells, after closing gaps of up to 8 cells within a unit.
(The control CA is one dense mass, without discrete units to count.)
Tips number far fewer than the units sampled at the final snapshot, since the units of a region trace back to a much smaller number of units at the early snapshot.


In [ ]:
def count_units(df: pd.DataFrame, key: str, dilation: int = 4) -> int:
    extant = df.loc[df["extant"], ["row", "col"]].to_numpy().astype(int)
    extant = extant - extant.min(axis=0) + dilation
    cells = np.zeros(extant.max(axis=0) + dilation + 1, dtype=bool)
    cells[tuple(extant.T)] = True
    labels, __ = scipy_ndimage.label(
        scipy_ndimage.binary_dilation(cells, iterations=dilation),
        structure=np.ones((3, 3)),
    )
    is_key = (df.loc[df["extant"], "position"] == key).to_numpy()
    return len(np.unique(labels[tuple(extant[is_key].T)]))


pd.DataFrame(
    {
        key: {
            "num final snapshot cells": (
                phylos["outlier"]["position"] == key
            ).sum(),
            "num final snapshot replicator units": count_units(
                phylos["outlier"], key
            ),
            "num persistent early snapshot tips": pfl.alifestd_count_leaf_nodes(
                prep_trimmed_tree(phylos["outlier"], key),
            ),
        }
        for key in ("sampled border", "sampled middle")
    },
)


## Summary

Observed imbalance of each tree, and where it falls within each null model.


In [ ]:
records = []
for name, (sampled, observed, leaf_node_counts) in null_samples.items():
    for key in sampled:
        null_df = pd.DataFrame(sampled[key])
        record = {
            "ca": name,
            "region": key,
            "num tips": (phylos[name]["position"] == key).sum(),
            "num tree leaves": leaf_node_counts[key],
            "observed colless": observed[key],
        }
        for source, group in null_df.groupby("source"):
            values = group["colless_index"]
            record[f"{source} null median"] = values.median()
            record[f"{source} null percentile"] = scipy_stats.percentileofscore(
                values, observed[key], kind="mean"
            )
        records.append(record)

pd.DataFrame(records).set_index(["ca", "region"]).T


Inner branch lengths of the trees, as a fraction of run length, and a Mann-Whitney U test of border against middle.


In [ ]:
records = []
for name, df in phylos.items():
    delta_cutoff_time = get_delta_cutoff_time(df)
    delta_df = prep_origin_time_delta(df)
    delta_df = delta_df[delta_df["origin_time"] < delta_cutoff_time]
    delta_df = delta_df[delta_df["origin_time_delta"].notna()]
    fracs = {
        src: group["origin_time_delta"] / df.attrs["num_steps"]
        for src, group in delta_df.groupby("source")
    }
    test = scipy_stats.mannwhitneyu(fracs["border"], fracs["middle"])
    records.append(
        {
            "ca": name,
            **{f"{src} num branches": len(v) for src, v in fracs.items()},
            **{f"{src} median length": v.median() for src, v in fracs.items()},
            **{f"{src} mean length": v.mean() for src, v in fracs.items()},
            "mann-whitney u": test.statistic,
            "p": test.pvalue,
        }
    )

pd.DataFrame(records).set_index("ca").T
